# P01 Mini-C4 Pretraining Corpus 源码全流程 Notebook

这本 notebook 不是通过 `subprocess` 调壳执行脚本，而是把 `project_1_mini_c4` 里的完整流程源码直接展开到 notebook 中。

你可以把它当成一本按项目顺序组织的代码讲义来阅读：

- 主流程脚本按推荐执行顺序排在前面
- 辅助脚本和工具脚本排在后面
- 如果 README 里提到某个脚本但仓库里缺失，也会保留缺口说明

## 项目环境

- 项目目录：`project_1_mini_c4`
- 建议 Conda 环境：`p1-mini-c4`

## 本 notebook 收录的源码文件顺序

1. `src/1_download_data.py` - 下载数据 [存在]
2. `src/2_process_warc.py` - 解析 WARC 并抽取正文 [存在]
3. `src/3_clean_data.py` - 基础清洗 [存在]
4. `src/4_deduplicate.py` - MinHash 去重 [存在]
5. `src/5_split_lang.py` - 语种拆分 [存在]
6. `src/6_quality_filter.py` - 质量过滤 [存在]
7. `src/7_prepare_training_data.py` - 封装训练数据 [存在]
8. `src/9_training_smoke_test.py` - 训练烟雾测试 [存在]
9. `src/8_evaluate_dataset.py` - 评估与报告 [存在]
10. `src/10_run_p1_checks.py` - 项目检查 [存在]
11. `src/pipeline_utils.py` - 辅助脚本 [存在]
12. `src/test_dedup_check.py` - 辅助脚本 [存在]

## 关键产物

- `data/processed/extracted_data.jsonl`
- `data/processed/clean_data.jsonl`
- `data/processed/deduplicated_data.jsonl`
- `data/processed/data_en.jsonl`
- `data/processed/data_zh.jsonl`
- `data/processed/final_data_en.jsonl`
- `data/processed/final_data_zh.jsonl`
- `data/processed/final_data.jsonl`
- `data/training/serialized_dataset.jsonl`
- `data/training/train.jsonl`
- `data/training/val.jsonl`
- `data/training/smoke_test.jsonl`
- `data/training/training_manifest.json`
- `data/reports/p1_metrics.json`
- `data/reports/p1_report.md`
- `data/reports/p1_test_results.json`
- `data/reports/p1_test_report.md`


## 项目 README

下面直接附上项目自带的 `README.md`，方便在 notebook 里连同源码一起看。

# P1 Mini-C4

This project builds a small, reproducible Mini-C4 style preprocessing pipeline on top of a Common Crawl shard.

## Environment

The project now has a reusable conda environment:

```bash
conda activate p1-mini-c4
```

Environment files:

- `environment.yml`: concise, project-facing dependency spec.
- `environment.lock.yml`: full export from the created `p1-mini-c4` environment for stricter reproduction.

The latest validation run was executed inside `p1-mini-c4`, not `base`.

## Deliverables

1. Project goals and scope
   - One WARC shard for a mini-scale dataset build.
   - Single-node CPU preprocessing with local models.
   - English/Chinese split plus final training-ready JSONL export.
2. Data retrieval and parsing
   - Download Common Crawl metadata and one WARC file.
   - Parse HTML responses and extract main text into JSONL.
3. Cleaning, deduplication, and quality filtering
   - Rule-based cleaning.
   - MinHash + LSH deduplication.
   - FastText language split.
   - Language-aware quality filtering with extra spam/menu/adult rejection rules.
4. Serialization and training preparation
   - Deterministic train/val split.
   - Serialized JSONL export with estimated token counts.
   - Train/val shard packaging.
   - Small smoke-test subset for downstream training checks.
5. Evaluation and cost analysis
   - Automatic metrics JSON and Markdown report.
   - Retention rates, language distribution, top domains, storage footprint, and optimization notes.
6. Extension directions
   - Multi-shard scaling, stronger domain filtering, Chinese LM scoring, and downstream tokenizer/pretraining reuse.

## Recommended Run Order

```bash
python src/1_download_data.py
python src/2_process_warc.py
python src/3_clean_data.py
python src/4_deduplicate.py
python src/5_split_lang.py
python src/6_quality_filter.py
python src/7_prepare_training_data.py
python src/9_training_smoke_test.py
python src/8_evaluate_dataset.py
python src/10_run_p1_checks.py
```

## Key Outputs

- `data/processed/extracted_data.jsonl`
- `data/processed/clean_data.jsonl`
- `data/processed/deduplicated_data.jsonl`
- `data/processed/data_en.jsonl`
- `data/processed/data_zh.jsonl`
- `data/processed/final_data_en.jsonl`
- `data/processed/final_data_zh.jsonl`
- `data/processed/final_data.jsonl`
- `data/training/serialized_dataset.jsonl`
- `data/training/train.jsonl`
- `data/training/val.jsonl`
- `data/training/smoke_test.jsonl`
- `data/training/training_manifest.json`
- `data/reports/p1_metrics.json`
- `data/reports/p1_report.md`
- `data/reports/p1_test_results.json`
- `data/reports/p1_test_report.md`

## Notes

- The current mini setup is intentionally small and suitable for local experimentation.
- `src/6_quality_filter.py` now rejects many mixed-language, navigation-heavy, repetitive, and adult/spam pages that previously leaked into the final corpus.
- `src/7_prepare_training_data.py` provides the training-facing packaging that was previously missing from P1.


## 源码目录概览

当前 `src/` 中实际存在的 Python 文件共 `12` 个：

- `src/10_run_p1_checks.py`
- `src/1_download_data.py`
- `src/2_process_warc.py`
- `src/3_clean_data.py`
- `src/4_deduplicate.py`
- `src/5_split_lang.py`
- `src/6_quality_filter.py`
- `src/7_prepare_training_data.py`
- `src/8_evaluate_dataset.py`
- `src/9_training_smoke_test.py`
- `src/pipeline_utils.py`
- `src/test_dedup_check.py`


## 完整源码展开


### `src/1_download_data.py` - 下载数据

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
import requests
import gzip
import os
from tqdm import tqdm

# ================= 配置部分 =================
# 1. 选择 Crawl ID (这是 2023 年第 50 周的抓取数据)
CRAWL_ID = "CC-MAIN-2023-50" 

# 2. 下载数量 (Mini项目建议 1-2 个，每个约 1GB)
NUM_FILES_TO_DOWNLOAD = 1

# 3. 数据保存目录
OUTPUT_DIR = "data/raw"

# Common Crawl 基础 URL
BASE_URL = "https://data.commoncrawl.org"
# ===========================================

def get_warc_file_paths(crawl_id, num_files):
    """
    获取指定 Crawl ID 的 WARC 文件下载路径列表
    """
    # 路径索引文件地址
    paths_url = f"{BASE_URL}/crawl-data/{crawl_id}/warc.paths.gz"
    print(f"📡 正在获取文件索引: {paths_url} ...")
    
    try:
        # 流式下载索引文件
        response = requests.get(paths_url, stream=True, timeout=10)
        response.raise_for_status()
        
        paths = []
        # 解压并读取前 num_files 行
        with gzip.open(response.raw, 'rt', encoding='utf-8') as f:
            for i, line in enumerate(f):
                if i >= num_files:
                    break
                paths.append(line.strip())
        return paths
    except Exception as e:
        print(f"❌ 获取索引失败: {e}")
        return []

def download_file(url, output_dir):
    """
    下载单个文件并显示进度条
    """
    local_filename = url.split('/')[-1]
    local_path = os.path.join(output_dir, local_filename)
    
    if os.path.exists(local_path):
        print(f"⚠️ 文件已存在，跳过: {local_filename}")
        return local_path

    print(f"⬇️ 开始下载: {local_filename}")
    
    try:
        # stream=True 确保不会一次性把 1GB 读入内存
        with requests.get(url, stream=True, timeout=30) as r:
            r.raise_for_status()
            total_size = int(r.headers.get('content-length', 0))
            
            with open(local_path, 'wb') as f, tqdm(
                desc=local_filename,
                total=total_size,
                unit='iB',
                unit_scale=True,
                unit_divisor=1024,
            ) as bar:
                for chunk in r.iter_content(chunk_size=8192):
                    size = f.write(chunk)
                    bar.update(size)
        print(f"✅ 下载完成: {local_path}")
        return local_path
    except Exception as e:
        print(f"❌ 下载失败 {url}: {e}")
        # 如果下载失败，删除损坏的半成品文件
        if os.path.exists(local_path):
            os.remove(local_path)
        return None

def main():
    # 确保输出目录存在
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)
    
    # 1. 获取文件路径列表
    warc_paths = get_warc_file_paths(CRAWL_ID, NUM_FILES_TO_DOWNLOAD)
    
    if not warc_paths:
        print("未找到文件路径，请检查网络或 CRAWL_ID。")
        return

    print(f"🎯 计划下载 {len(warc_paths)} 个文件到 {OUTPUT_DIR} ...")

    # 2. 循环下载
    for relative_path in warc_paths:
        full_url = f"{BASE_URL}/{relative_path}"
        download_file(full_url, OUTPUT_DIR)
    
    print("\n🎉 数据准备阶段完成！")

if __name__ == "__main__":
    main()


### `src/2_process_warc.py` - 解析 WARC 并抽取正文

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
import json
import os

from tqdm import tqdm
import trafilatura
from warcio.archiveiterator import ArchiveIterator


# ================= 配置部分 =================
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)

RAW_DIR = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

WARC_FILENAME = "CC-MAIN-2023-50-segment-1700679099281.0-1700679117904.warc.gz"
INPUT_FILE = os.path.join(RAW_DIR, WARC_FILENAME)
OUTPUT_FILE = os.path.join(PROCESSED_DIR, "extracted_data.jsonl")

# Mini 版本默认只处理前 10000 条记录，便于快速复现。
LIMIT_RECORDS = 10000
# ===========================================


def extract_text_from_warc(warc_path, output_path, limit=None):
    """
    读取 WARC 文件，提取正文，并保存为 JSONL。
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    print(f"🚀 开始处理: {warc_path}")
    print(f"💾 输出结果: {output_path}")

    counter = 0
    success_count = 0

    with open(output_path, "w", encoding="utf-8") as out_f:
        with open(warc_path, "rb") as stream:
            for record in tqdm(ArchiveIterator(stream), desc="Processing Records"):
                if record.rec_type != "response":
                    counter += 1
                    if limit and counter >= limit:
                        break
                    continue

                content_type = record.http_headers.get_header("Content-Type")
                if not content_type or "text/html" not in content_type:
                    counter += 1
                    if limit and counter >= limit:
                        break
                    continue

                try:
                    content = record.content_stream().read()
                except Exception:
                    counter += 1
                    if limit and counter >= limit:
                        break
                    continue

                text = trafilatura.extract(
                    content,
                    include_comments=False,
                    include_tables=False,
                    no_fallback=False,
                )

                if text and len(text.strip()) > 0:
                    url = record.rec_headers.get_header("WARC-Target-URI")
                    data = {
                        "url": url,
                        "text": text,
                    }
                    out_f.write(json.dumps(data, ensure_ascii=False) + "\n")
                    success_count += 1

                counter += 1
                if limit and counter >= limit:
                    break

    print("\n✅ 处理完成！")
    print(f"📊 扫描记录数: {counter}")
    print(f"📄 成功提取数: {success_count}")


def main():
    input_path = INPUT_FILE
    if not os.path.exists(input_path):
        files = [f for f in os.listdir(RAW_DIR) if f.endswith(".warc.gz")]
        if files:
            input_path = os.path.join(RAW_DIR, files[0])
            print(f"自动发现文件: {input_path}")
        else:
            print(f"❌ 错误: 找不到输入文件 {INPUT_FILE}，且目录下没有其他 .warc.gz 文件")
            return

    extract_text_from_warc(input_path, OUTPUT_FILE, LIMIT_RECORDS)


if __name__ == "__main__":
    main()


### `src/3_clean_data.py` - 基础清洗

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
import json
import os
from tqdm import tqdm
import re

# ================= 配置部分 =================
# 1. 自动定位路径 (保持和你之前的习惯一致)
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

# 输入文件 (上一步的输出)
INPUT_FILE = os.path.join(DATA_DIR, "extracted_data.jsonl")
# 输出文件 (清洗后的文件)
OUTPUT_FILE = os.path.join(DATA_DIR, "clean_data.jsonl")

# ================= 核心清洗规则 =================
def is_high_quality(text):
    """
    判断文本是否“干净”。返回 True 表示保留，False 表示丢弃。
    """
    
    # --- 规则 1: 长度过滤 ---
    # 太短通常是导航、版权声明；太长可能是日志堆栈
    if len(text) < 100 or len(text) > 2_000_000:
        return False
        
    # --- 规则 2: 平均词长 (Gopher/C4 经典规则) ---
    # 正常的英语/中文文本，平均词长通常在一定范围内。
    # 如果全是 "a b c d" (过短) 或 "supercalifragilistic..." (过长/代码)，则是垃圾
    words = text.split()
    if len(words) == 0:
        return False
    mean_word_len = sum(len(w) for w in words) / len(words)
    
    # 英文通常 3-10 是正常范围；中文分词后略有不同，但作为粗筛
    # 如果平均每个词超过 15 个字符，通常是 URL 列表或乱码代码
    if mean_word_len > 15:
        return False

    # --- 规则 3: 符号密度 (Symbol Ratio) ---
    # 如果一句话里包含大量的 '{', '}', '<', '>', '#', 通常是代码或 JSON 数据
    # 我们检查常用代码符号的占比
    code_symbols = {'{', '}', '[', ']', '<', '>', '\\'}
    symbol_count = sum(1 for char in text if char in code_symbols)
    
    # 如果代码符号占比超过 10%，视为垃圾
    if symbol_count / len(text) > 0.1:
        return False

    # --- 规则 4: 关键词黑名单 (Blocklist) ---
    # 常见的系统报错、Cookie 提示、Lorem Ipsum
    bad_phrases = [
        "lorem ipsum", 
        "javascript is disabled", 
        "enable cookies",
        "403 forbidden",
        "404 not found",
        "access denied",
        "rights reserved" # 只有版权声明的短文本
    ]
    
    # 转换为小写进行快速检查
    text_lower = text.lower()
    for phrase in bad_phrases:
        if phrase in text_lower:
            # 如果文本很短且包含脏词，直接删掉
            # 如果文本很长但包含脏词，可能是正文里引用了，这里为了严格，简单粗暴处理：
            # 策略：如果文本短于 500 字且包含脏词 -> 删
            if len(text) < 500:
                return False
    
    return True

# ================= 主流程 =================
def main():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 找不到输入文件: {INPUT_FILE}")
        return

    print(f"🚀 开始清洗: {INPUT_FILE}")
    print(f"💾 输出位置: {OUTPUT_FILE}")

    stats = {
        "total": 0,
        "kept": 0,
        "dropped": 0
    }

    with open(INPUT_FILE, 'r', encoding='utf-8') as f_in, \
         open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        
        for line in tqdm(f_in, desc="Cleaning Data"):
            stats["total"] += 1
            try:
                item = json.loads(line)
                text = item.get("text", "")
                
                # 调用清洗函数
                if is_high_quality(text):
                    f_out.write(line) # 原样写入，或者只写入 text
                    stats["kept"] += 1
                else:
                    stats["dropped"] += 1
                    
            except json.JSONDecodeError:
                continue

    print("\n✅ 清洗完成！")
    print(f"📊 总数: {stats['total']}")
    print(f"🗑️ 丢弃: {stats['dropped']} ({(stats['dropped']/stats['total'])*100:.2f}%)")
    print(f"💎 保留: {stats['kept']}")

if __name__ == "__main__":
    main()


### `src/4_deduplicate.py` - MinHash 去重

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
import ray
import json
import os
from datasketch import MinHash, MinHashLSH
from tqdm import tqdm
import time

# ================= 配置 =================
# 自动设置路径
CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

INPUT_FILE = os.path.join(DATA_DIR, "clean_data.jsonl")  # 上一步清洗完的文件
OUTPUT_FILE = os.path.join(DATA_DIR, "deduplicated_data.jsonl")

# MinHash 参数 (C4 标准参数: num_perm=128)
NUM_PERM = 128 
THRESHOLD = 0.8  # 相似度阈值，超过 0.8 视为重复

# ================= Ray Actor =================
# 我们初始化 Ray，利用单机所有 CPU 核心
ray.init(ignore_reinit_error=True)

def get_minhash(text, num_perm=128):
    """
    计算一段文本的 MinHash 签名
    """
    m = MinHash(num_perm=num_perm)
    # 使用简单的 shingle (按单词分)
    words = text.split()
    for w in words:
        m.update(w.encode('utf8'))
    return m

@ray.remote
def process_batch(lines, batch_id):
    """
    Ray Worker: 处理一批数据，计算 MinHash
    返回: List of (url, minhash_obj, text_content)
    """
    results = []
    for line in lines:
        try:
            item = json.loads(line)
            url = item['url']
            text = item['text']
            
            # 计算签名
            minhash = get_minhash(text, NUM_PERM)
            results.append((url, minhash, text))
        except Exception:
            continue
    return results

# ================= 主流程 =================
def main():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 找不到文件: {INPUT_FILE}")
        return

    print("🚀 第一阶段: 并行计算 MinHash 签名...")
    
    # 1. 读取所有数据并分批 (Batching)
    # 为了避免内存爆炸，我们按块读取，但为了演示简单，这里假设内存够大
    # 实际上应该用 Ray Dataset 或者流式读取，这里用简易版分批
    batch_size = 1000
    all_lines = []
    
    # 读取文件 (如果文件有几十G，不要这样读，要用流式)
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        all_lines = f.readlines()
    
    total_records = len(all_lines)
    print(f"📚 总记录数: {total_records}")

    # 将数据切分成小块，分发给 Ray
    batches = [all_lines[i:i + batch_size] for i in range(0, total_records, batch_size)]
    
    # 提交任务给 Ray (非阻塞)
    futures = [process_batch.remote(batch, i) for i, batch in enumerate(batches)]
    
    # 获取结果 (阻塞等待所有 CPU 算完)
    print("⏳ 等待 CPU 计算中 ")
    processed_batches = ray.get(futures)
    
    # 展平结果
    # results 结构: [(url, minhash, text), (url, minhash, text), ...]
    results = [item for batch in processed_batches for item in batch]

    
    print("\n🚀 第二阶段: 构建 LSH 索引并去重...")
    # 这一步通常难以并行化，必须在主进程构建全局索引
    # 就像查字典一样，必须有一本完整的字典
    
    lsh = MinHashLSH(threshold=THRESHOLD, num_perm=NUM_PERM)
    
    unique_records = []
    duplicate_count = 0
    
    # 开始遍历并查重
    for url, minhash, text in tqdm(results, desc="LSH Deduplication"):
        # 查询 LSH 桶里是否有相似的
        # query 返回的是已经存在于桶里的 key (这里我们用 url 当 key)
        duplicates = lsh.query(minhash)
        
        if len(duplicates) > 0:
            # 发现重复！
            duplicate_count += 1
            # 策略：简单的丢弃当前这条，保留桶里那条
            # 你也可以根据时间戳保留最新的，这里从简
        else:
            # 没有重复，插入桶中
            lsh.insert(url, minhash)
            unique_records.append({"url": url, "text": text})

    print(f"\n✅ 去重完成！")
    print(f"🗑️ 发现重复: {duplicate_count}")
    print(f"💎 剩余有效: {len(unique_records)}")
    
    # 保存结果
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        for item in unique_records:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')

    ray.shutdown()

if __name__ == "__main__":
    main()


### `src/5_split_lang.py` - 语种拆分

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import os

import fasttext

from pipeline_utils import normalize_text

CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODEL_PATH = os.path.join(PROJECT_ROOT, "models", "lid.176.ftz")

INPUT_FILE = os.path.join(DATA_DIR, "deduplicated_data.jsonl")
OUTPUT_FILES = {
    "en": os.path.join(DATA_DIR, "data_en.jsonl"),
    "zh": os.path.join(DATA_DIR, "data_zh.jsonl"),
    "others": os.path.join(DATA_DIR, "data_others.jsonl"),
}

MIN_TEXT_CHARS = 40


def main() -> None:
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 找不到输入文件: {INPUT_FILE}")
        return

    if not os.path.exists(MODEL_PATH):
        print(f"❌ 找不到 FastText 模型: {MODEL_PATH}")
        return

    model = fasttext.load_model(MODEL_PATH)
    stats = {"total": 0, "en": 0, "zh": 0, "others": 0, "skipped": 0}

    with open(INPUT_FILE, "r", encoding="utf-8") as f_in, \
         open(OUTPUT_FILES["en"], "w", encoding="utf-8") as f_en, \
         open(OUTPUT_FILES["zh"], "w", encoding="utf-8") as f_zh, \
         open(OUTPUT_FILES["others"], "w", encoding="utf-8") as f_others:

        writers = {"en": f_en, "zh": f_zh, "others": f_others}

        for line in f_in:
            try:
                data = json.loads(line)
            except json.JSONDecodeError:
                stats["skipped"] += 1
                continue

            text = normalize_text(data.get("text", ""))
            if len(text) < MIN_TEXT_CHARS:
                stats["skipped"] += 1
                continue

            labels, scores = model.predict(text.replace("\n", " "), k=1)
            detected_lang = labels[0].replace("__label__", "")
            confidence = float(scores[0])

            data["text"] = text
            data["detected_lang"] = detected_lang
            data["lang_confidence"] = round(confidence, 6)

            if detected_lang == "en":
                bucket = "en"
            elif detected_lang == "zh":
                bucket = "zh"
            else:
                bucket = "others"

            writers[bucket].write(json.dumps(data, ensure_ascii=False) + "\n")

            stats["total"] += 1
            stats[bucket] += 1

    print("语言切分完成！")
    print(json.dumps(stats, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### `src/6_quality_filter.py` - 质量过滤

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import os

from tqdm import tqdm

from pipeline_utils import (
    character_ratio,
    duplicate_line_ratio,
    estimate_token_count,
    median_line_length,
    normalize_text,
    script_counts,
    sha1_text,
    short_line_ratio,
    split_nonempty_lines,
)

try:
    import kenlm
except ImportError:
    kenlm = None


CURRENT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(CURRENT_DIR)
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
MODEL_DIR = os.path.join(PROJECT_ROOT, "models")

INPUT_FILES = {
    "en": os.path.join(DATA_DIR, "data_en.jsonl"),
    "zh": os.path.join(DATA_DIR, "data_zh.jsonl"),
}
OUTPUT_FILES = {
    "en": os.path.join(DATA_DIR, "final_data_en.jsonl"),
    "zh": os.path.join(DATA_DIR, "final_data_zh.jsonl"),
}
MERGED_OUTPUT_FILE = os.path.join(DATA_DIR, "final_data.jsonl")
STATS_FILE = os.path.join(DATA_DIR, "quality_filter_stats.json")
MODEL_PATH = os.path.join(MODEL_DIR, "en.arpa.bin")

PERPLEXITY_THRESHOLD = -6.0
MIN_CHARS = {"en": 160, "zh": 120}

NAVIGATION_KEYWORDS = {
    "home", "about", "contact", "privacy", "terms", "copyright",
    "login", "register", "search", "menu", "help", "forum", "news",
    "首页", "登录", "注册", "联系我们", "关于我们", "隐私", "版权", "返回顶部",
    "目录", "搜索", "帮助", "校历", "课程表", "Contact Us",
}
ADULT_SPAM_PHRASES = [
    "自拍偷拍", "裸聊", "直播平台", "无码", "成人", "麻豆", "约炮", "台灣uu",
    "真愛旅舍", "大尺度", "美女直播", "性色", "性聊", "91视频", "色情",
    "casino", "sportsbook", "poker", "xxx", "porn", "escort",
]


def is_navigation_line(line: str) -> bool:
    lower = line.lower()
    if len(line) <= 32 and any(keyword.lower() in lower for keyword in NAVIGATION_KEYWORDS):
        return True
    if len(line) <= 48 and (line.count("|") >= 2 or line.count("/") >= 4):
        return True
    if "all rights reserved" in lower or "copyright" in lower:
        return True
    return False


def reject_reason(text: str, lang: str, lm_model) -> tuple[str | None, dict]:
    normalized = normalize_text(text)
    if not normalized:
        return "empty", {}

    lines = split_nonempty_lines(normalized)
    counts = script_counts(normalized)
    estimated_tokens = estimate_token_count(normalized)
    nav_ratio = (
        sum(1 for line in lines if is_navigation_line(line)) / len(lines) if lines else 0.0
    )
    duplicate_ratio = duplicate_line_ratio(lines)
    short_ratio = short_line_ratio(lines)
    median_len = median_line_length(lines)
    digit_ratio = character_ratio(normalized, "digits")

    metadata = {
        "estimated_tokens": estimated_tokens,
        "script_counts": counts,
        "line_count": len(lines),
        "duplicate_line_ratio": round(duplicate_ratio, 4),
        "navigation_line_ratio": round(nav_ratio, 4),
    }

    if len(normalized) < MIN_CHARS[lang]:
        return "too_short", metadata
    if estimated_tokens < 40:
        return "too_few_tokens", metadata
    if duplicate_ratio > 0.2:
        return "duplicate_lines", metadata
    if len(lines) >= 12 and short_ratio > 0.75 and median_len < 28:
        return "directory_like", metadata
    if nav_ratio > 0.35:
        return "navigation_heavy", metadata
    if digit_ratio > 0.2:
        return "digit_heavy", metadata

    lowered = normalized.lower()
    adult_hits = sum(1 for phrase in ADULT_SPAM_PHRASES if phrase.lower() in lowered)
    if adult_hits >= 2:
        return "adult_or_spam", metadata

    if lang == "en":
        if counts["latin"] < 120:
            return "insufficient_english", metadata
        if counts["cjk"] > max(40, counts["latin"] * 0.1):
            return "mixed_language_en", metadata
        if lm_model is not None:
            words = max(1, len(normalized.split()))
            normalized_score = lm_model.score(normalized) / words
            metadata["perplexity_score"] = round(normalized_score, 6)
            if normalized_score <= PERPLEXITY_THRESHOLD:
                return "low_kenlm_score", metadata
    elif lang == "zh":
        if counts["cjk"] < 80:
            return "insufficient_chinese", metadata
        if counts["latin"] > counts["cjk"]:
            return "mixed_language_zh", metadata

    return None, metadata


def filter_file(lang: str, input_path: str, output_path: str, lm_model) -> tuple[dict, list[dict]]:
    stats = {
        "total": 0,
        "kept": 0,
        "dropped": 0,
        "reasons": {},
    }
    kept_records = []
    seen_hashes = set()

    if not os.path.exists(input_path):
        return stats, kept_records

    with open(input_path, "r", encoding="utf-8") as f_in, \
         open(output_path, "w", encoding="utf-8") as f_out:

        for line in tqdm(f_in, desc=f"Filtering {lang}"):
            stats["total"] += 1
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                stats["dropped"] += 1
                stats["reasons"]["json_error"] = stats["reasons"].get("json_error", 0) + 1
                continue

            text = normalize_text(item.get("text", ""))
            reason, metadata = reject_reason(text, lang, lm_model)
            if reason is not None:
                stats["dropped"] += 1
                stats["reasons"][reason] = stats["reasons"].get(reason, 0) + 1
                continue

            text_sha1 = sha1_text(text)
            if text_sha1 in seen_hashes:
                stats["dropped"] += 1
                stats["reasons"]["exact_duplicate"] = stats["reasons"].get("exact_duplicate", 0) + 1
                continue
            seen_hashes.add(text_sha1)

            item["text"] = text
            item["lang"] = lang
            item["text_sha1"] = text_sha1
            for key, value in metadata.items():
                item[key] = value

            f_out.write(json.dumps(item, ensure_ascii=False) + "\n")
            kept_records.append(item)
            stats["kept"] += 1

    return stats, kept_records


def main() -> None:
    lm_model = None
    if kenlm is not None and os.path.exists(MODEL_PATH):
        print(f"🚀 加载 KenLM 模型: {MODEL_PATH}")
        lm_model = kenlm.Model(MODEL_PATH)

    all_stats = {}
    merged_records = []

    for lang in ("en", "zh"):
        stats, kept_records = filter_file(
            lang=lang,
            input_path=INPUT_FILES[lang],
            output_path=OUTPUT_FILES[lang],
            lm_model=lm_model if lang == "en" else None,
        )
        all_stats[lang] = stats
        merged_records.extend(kept_records)

    with open(MERGED_OUTPUT_FILE, "w", encoding="utf-8") as f:
        for item in merged_records:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    with open(STATS_FILE, "w", encoding="utf-8") as f:
        json.dump(all_stats, f, ensure_ascii=False, indent=2)

    print("质量过滤完成！")
    print(json.dumps(all_stats, ensure_ascii=False, indent=2))
    print(f"💾 合并输出: {MERGED_OUTPUT_FILE}")


if __name__ == "__main__":
    main()


### `src/7_prepare_training_data.py` - 封装训练数据

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

from pipeline_utils import estimate_token_count, normalize_text, sha1_text

CURRENT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = CURRENT_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_DIR = DATA_DIR / "training"

INPUT_FILE = PROCESSED_DIR / "final_data.jsonl"
SERIALIZED_FILE = TRAINING_DIR / "serialized_dataset.jsonl"
TRAIN_FILE = TRAINING_DIR / "train.jsonl"
VAL_FILE = TRAINING_DIR / "val.jsonl"
SMOKE_TEST_FILE = TRAINING_DIR / "smoke_test.jsonl"
MANIFEST_FILE = TRAINING_DIR / "training_manifest.json"

TRAIN_DIR = TRAINING_DIR / "train_shards"
VAL_DIR = TRAINING_DIR / "val_shards"

VAL_RATIO_PERCENT = 10
SHARD_SIZE = 256
SMOKE_TEST_SIZE = 32


def assign_split(text_sha1: str) -> str:
    bucket = int(text_sha1[:8], 16) % 100
    return "val" if bucket < VAL_RATIO_PERCENT else "train"


def write_jsonl(path: Path, records: list[dict]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def write_shards(records: list[dict], output_dir: Path, prefix: str) -> list[str]:
    output_dir.mkdir(parents=True, exist_ok=True)
    written = []
    for shard_idx, start in enumerate(range(0, len(records), SHARD_SIZE)):
        shard_records = records[start:start + SHARD_SIZE]
        shard_path = output_dir / f"{prefix}-{shard_idx:05d}.jsonl"
        write_jsonl(shard_path, shard_records)
        written.append(str(shard_path.relative_to(PROJECT_ROOT)))
    return written


def build_records() -> list[dict]:
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Missing input file: {INPUT_FILE}")

    records = []
    seen_hashes = set()
    with INPUT_FILE.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue

            text = normalize_text(item.get("text", ""))
            if not text:
                continue

            text_sha1 = sha1_text(text)
            if text_sha1 in seen_hashes:
                continue
            seen_hashes.add(text_sha1)

            record = {
                "id": f"mini_c4_{len(records):06d}",
                "source_line": line_number,
                "url": item.get("url"),
                "lang": item.get("lang", "unknown"),
                "text": text,
                "text_sha1": text_sha1,
                "n_chars": len(text),
                "estimated_tokens": estimate_token_count(text),
            }
            record["split"] = assign_split(text_sha1)
            records.append(record)

    return records


def validate_records(train_records: list[dict], val_records: list[dict]) -> dict:
    train_hashes = {record["text_sha1"] for record in train_records}
    val_hashes = {record["text_sha1"] for record in val_records}
    overlap = train_hashes & val_hashes
    if overlap:
        raise ValueError(f"Train/val overlap detected: {len(overlap)} items")

    if any(not record["text"].strip() for record in train_records + val_records):
        raise ValueError("Empty text detected after serialization")

    return {
        "train_hashes": len(train_hashes),
        "val_hashes": len(val_hashes),
        "overlap": len(overlap),
    }


def build_smoke_test_records(train_records: list[dict]) -> list[dict]:
    by_lang = {}
    for record in train_records:
        by_lang.setdefault(record["lang"], []).append(record)

    smoke_test = []
    lang_keys = sorted(by_lang.keys())
    index_by_lang = {lang: 0 for lang in lang_keys}

    while len(smoke_test) < min(SMOKE_TEST_SIZE, len(train_records)):
        made_progress = False
        for lang in lang_keys:
            current_index = index_by_lang[lang]
            if current_index >= len(by_lang[lang]):
                continue
            smoke_test.append(by_lang[lang][current_index])
            index_by_lang[lang] += 1
            made_progress = True
            if len(smoke_test) >= SMOKE_TEST_SIZE:
                break
        if not made_progress:
            break

    return smoke_test


def main() -> None:
    TRAINING_DIR.mkdir(parents=True, exist_ok=True)

    records = build_records()
    train_records = [record for record in records if record["split"] == "train"]
    val_records = [record for record in records if record["split"] == "val"]
    smoke_test_records = build_smoke_test_records(train_records)

    validation = validate_records(train_records, val_records)

    write_jsonl(SERIALIZED_FILE, records)
    write_jsonl(TRAIN_FILE, train_records)
    write_jsonl(VAL_FILE, val_records)
    write_jsonl(SMOKE_TEST_FILE, smoke_test_records)

    train_shards = write_shards(train_records, TRAIN_DIR, "train")
    val_shards = write_shards(val_records, VAL_DIR, "val")

    lang_distribution = {}
    for record in records:
        lang_distribution[record["lang"]] = lang_distribution.get(record["lang"], 0) + 1

    manifest = {
        "input_file": str(INPUT_FILE.relative_to(PROJECT_ROOT)),
        "serialized_file": str(SERIALIZED_FILE.relative_to(PROJECT_ROOT)),
        "train_file": str(TRAIN_FILE.relative_to(PROJECT_ROOT)),
        "val_file": str(VAL_FILE.relative_to(PROJECT_ROOT)),
        "smoke_test_file": str(SMOKE_TEST_FILE.relative_to(PROJECT_ROOT)),
        "num_records": len(records),
        "num_train_records": len(train_records),
        "num_val_records": len(val_records),
        "num_smoke_test_records": len(smoke_test_records),
        "estimated_tokens_total": sum(record["estimated_tokens"] for record in records),
        "estimated_tokens_train": sum(record["estimated_tokens"] for record in train_records),
        "estimated_tokens_val": sum(record["estimated_tokens"] for record in val_records),
        "lang_distribution": lang_distribution,
        "validation": validation,
        "train_shards": train_shards,
        "val_shards": val_shards,
    }

    with MANIFEST_FILE.open("w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print("Training data preparation complete.")
    print(json.dumps(manifest, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### `src/9_training_smoke_test.py` - 训练烟雾测试

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

CURRENT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = CURRENT_DIR.parent
SMOKE_TEST_FILE = PROJECT_ROOT / "data" / "training" / "smoke_test.jsonl"


def main() -> None:
    if not SMOKE_TEST_FILE.exists():
        raise FileNotFoundError(
            f"Smoke test file not found: {SMOKE_TEST_FILE}. Run 7_prepare_training_data.py first."
        )

    records = []
    ids = set()
    with SMOKE_TEST_FILE.open("r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            records.append(item)
            if item["id"] in ids:
                raise ValueError(f"Duplicate id detected in smoke test: {item['id']}")
            ids.add(item["id"])
            if not item.get("text", "").strip():
                raise ValueError(f"Empty text detected in smoke test record: {item['id']}")
            if item.get("estimated_tokens", 0) <= 0:
                raise ValueError(f"Non-positive token estimate in smoke test record: {item['id']}")

    summary = {
        "num_records": len(records),
        "languages": sorted({record.get("lang", "unknown") for record in records}),
        "min_tokens": min(record["estimated_tokens"] for record in records) if records else 0,
        "max_tokens": max(record["estimated_tokens"] for record in records) if records else 0,
    }

    print("Training smoke test passed.")
    print(json.dumps(summary, ensure_ascii=False, indent=2))


if __name__ == "__main__":
    main()


### `src/8_evaluate_dataset.py` - 评估与报告

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

from pipeline_utils import estimate_token_count, extract_domain

CURRENT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = CURRENT_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_DIR = DATA_DIR / "training"
REPORT_DIR = DATA_DIR / "reports"

FILES = {
    "extracted": PROCESSED_DIR / "extracted_data.jsonl",
    "clean": PROCESSED_DIR / "clean_data.jsonl",
    "deduplicated": PROCESSED_DIR / "deduplicated_data.jsonl",
    "lang_en": PROCESSED_DIR / "data_en.jsonl",
    "lang_zh": PROCESSED_DIR / "data_zh.jsonl",
    "final": PROCESSED_DIR / "final_data.jsonl",
}

QUALITY_STATS_FILE = PROCESSED_DIR / "quality_filter_stats.json"
TRAINING_MANIFEST_FILE = TRAINING_DIR / "training_manifest.json"
METRICS_JSON = REPORT_DIR / "p1_metrics.json"
REPORT_MD = REPORT_DIR / "p1_report.md"


def count_lines(path: Path) -> int:
    if not path.exists():
        return 0
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


def file_size_mb(path: Path) -> float:
    if not path.exists():
        return 0.0
    return round(path.stat().st_size / (1024 * 1024), 2)


def directory_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    total_bytes = 0
    for child in path.rglob("*"):
        if child.is_file():
            total_bytes += child.stat().st_size
    return round(total_bytes / (1024 ** 3), 2)


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def summarize_final_dataset(path: Path) -> dict:
    language_counter = Counter()
    domain_counter = Counter()
    doc_count = 0
    char_count = 0
    token_count = 0

    if not path.exists():
        return {
            "doc_count": 0,
            "char_count": 0,
            "estimated_tokens": 0,
            "avg_chars_per_doc": 0.0,
            "avg_tokens_per_doc": 0.0,
            "languages": {},
            "top_domains": [],
        }

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue
            text = item.get("text", "")
            doc_count += 1
            char_count += len(text)
            token_count += estimate_token_count(text)
            language_counter[item.get("lang", "unknown")] += 1
            domain_counter[extract_domain(item.get("url", ""))] += 1

    return {
        "doc_count": doc_count,
        "char_count": char_count,
        "estimated_tokens": token_count,
        "avg_chars_per_doc": round(char_count / doc_count, 2) if doc_count else 0.0,
        "avg_tokens_per_doc": round(token_count / doc_count, 2) if doc_count else 0.0,
        "languages": dict(language_counter),
        "top_domains": domain_counter.most_common(10),
    }


def build_metrics() -> dict:
    stage_counts = {name: count_lines(path) for name, path in FILES.items()}
    stage_sizes_mb = {name: file_size_mb(path) for name, path in FILES.items()}
    final_summary = summarize_final_dataset(FILES["final"])
    quality_stats = load_json(QUALITY_STATS_FILE)
    training_manifest = load_json(TRAINING_MANIFEST_FILE)

    extracted_count = stage_counts.get("extracted", 0) or 1
    clean_count = stage_counts.get("clean", 0)
    dedup_count = stage_counts.get("deduplicated", 0)
    final_count = stage_counts.get("final", 0)

    retention = {
        "clean_over_extracted": round(clean_count / extracted_count, 4),
        "dedup_over_extracted": round(dedup_count / extracted_count, 4),
        "final_over_extracted": round(final_count / extracted_count, 4),
    }

    raw_disk_gb = directory_size_gb(DATA_DIR / "raw")
    processed_disk_gb = directory_size_gb(PROCESSED_DIR)
    model_disk_gb = directory_size_gb(PROJECT_ROOT / "models")
    total_disk_gb = round(raw_disk_gb + processed_disk_gb + model_disk_gb, 2)
    recommended_free_disk_gb = round(total_disk_gb * 1.5, 2)
    monthly_storage_cost_usd = round(total_disk_gb * 0.023, 2)

    return {
        "stage_counts": stage_counts,
        "stage_sizes_mb": stage_sizes_mb,
        "retention": retention,
        "final_summary": final_summary,
        "quality_stats": quality_stats,
        "training_manifest": training_manifest,
        "hardware_boundary": {
            "raw_disk_gb": raw_disk_gb,
            "processed_disk_gb": processed_disk_gb,
            "model_disk_gb": model_disk_gb,
            "total_disk_gb": total_disk_gb,
            "recommended_free_disk_gb": recommended_free_disk_gb,
            "execution_mode": "single-node CPU preprocessing; Ray for MinHash dedup; KenLM for English quality scoring",
        },
        "cost_analysis": {
            "monthly_storage_cost_usd_estimate": monthly_storage_cost_usd,
            "download_bandwidth_gb": raw_disk_gb,
            "notes": [
                "Common Crawl download is free, but local bandwidth and storage are the dominant dataset-building costs.",
                "Model footprint is larger than processed data footprint, so quality-filter models should be shared across experiments.",
                "The current mini scope is suitable for laptop or single-server CPU preprocessing.",
            ],
            "optimization_space": [
                "Replace full-file reads in dedup with streaming or chunked indexing for larger crawls.",
                "Add domain allow/block lists before language split to reduce wasted parsing.",
                "Train or import a Chinese LM for zh quality scoring to reduce mixed-language leakage.",
                "Persist intermediate metrics per stage to measure runtime, not just data volume.",
            ],
        },
    }


def render_report(metrics: dict) -> str:
    stage_counts = metrics["stage_counts"]
    retention = metrics["retention"]
    final_summary = metrics["final_summary"]
    hardware = metrics["hardware_boundary"]
    cost = metrics["cost_analysis"]
    training = metrics.get("training_manifest", {})

    top_domains_lines = "\n".join(
        f"- {domain}: {count}" for domain, count in final_summary.get("top_domains", [])
    ) or "- None"

    return f"""# P1 Mini-C4 Delivery Report

## 1. Project Goals And Data Scope

- Goal: build a reproducible Mini-C4 style preprocessing pipeline on a small Common Crawl slice.
- Scope: single WARC file, CPU-only local preprocessing, English/Chinese split, final training-ready JSONL export.
- Hardware boundary: raw {hardware['raw_disk_gb']} GB, processed {hardware['processed_disk_gb']} GB, models {hardware['model_disk_gb']} GB, recommended free disk >= {hardware['recommended_free_disk_gb']} GB.

## 2. Data Retrieval And Parsing

- Extracted records: {stage_counts.get('extracted', 0)}
- Cleaned records: {stage_counts.get('clean', 0)}
- Deduplicated records: {stage_counts.get('deduplicated', 0)}
- Final records: {stage_counts.get('final', 0)}
- Pipeline path: download -> WARC parse -> cleaning -> MinHash dedup -> language split -> quality filtering.

## 3. Cleaning, Deduplication And Quality Filtering

- Clean retention: {retention['clean_over_extracted']:.2%}
- Dedup retention: {retention['dedup_over_extracted']:.2%}
- Final retention: {retention['final_over_extracted']:.2%}
- Final language distribution: {json.dumps(final_summary.get('languages', {}), ensure_ascii=False)}
- Top final domains:
{top_domains_lines}

## 4. Serialization And Training Preparation

- Serialized dataset records: {training.get('num_records', 0)}
- Train records: {training.get('num_train_records', 0)}
- Val records: {training.get('num_val_records', 0)}
- Smoke test records: {training.get('num_smoke_test_records', 0)}
- Estimated tokens (train): {training.get('estimated_tokens_train', 0)}
- Outputs: `serialized_dataset.jsonl`, `train.jsonl`, `val.jsonl`, shard files, `smoke_test.jsonl`, and `training_manifest.json`.

## 5. Evaluation And Cost Analysis

- Avg chars/doc: {final_summary.get('avg_chars_per_doc', 0)}
- Avg estimated tokens/doc: {final_summary.get('avg_tokens_per_doc', 0)}
- Estimated monthly storage cost at $0.023/GB-month: ${cost['monthly_storage_cost_usd_estimate']}
- Dominant cost structure: disk footprint, download bandwidth, and CPU-based filtering/models.

## 6. Extension Directions

- Scale from one WARC shard to multiple shards with chunked dedup and per-stage runtime logs.
- Add stronger domain filtering before parse to reduce obvious spam/adult/menu pages.
- Add Chinese LM scoring and tokenizer-aware packaging for multilingual pretraining.
- Connect downstream chapters by reusing `data/training/` artifacts as tokenizer or pretraining inputs.
"""


def main() -> None:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    metrics = build_metrics()

    with METRICS_JSON.open("w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    report = render_report(metrics)
    with REPORT_MD.open("w", encoding="utf-8") as f:
        f.write(report)

    print("Dataset evaluation complete.")
    print(report)


if __name__ == "__main__":
    main()


### `src/10_run_p1_checks.py` - 项目检查

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from collections import Counter
from datetime import datetime, UTC
from pathlib import Path


CURRENT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = CURRENT_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
TRAINING_DIR = DATA_DIR / "training"
REPORT_DIR = DATA_DIR / "reports"

RESULTS_JSON = REPORT_DIR / "p1_test_results.json"
RESULTS_MD = REPORT_DIR / "p1_test_report.md"

SCRIPT_FILES = [
    CURRENT_DIR / "pipeline_utils.py",
    CURRENT_DIR / "1_download_data.py",
    CURRENT_DIR / "2_process_warc.py",
    CURRENT_DIR / "3_clean_data.py",
    CURRENT_DIR / "4_deduplicate.py",
    CURRENT_DIR / "5_split_lang.py",
    CURRENT_DIR / "6_quality_filter.py",
    CURRENT_DIR / "7_prepare_training_data.py",
    CURRENT_DIR / "8_evaluate_dataset.py",
    CURRENT_DIR / "9_training_smoke_test.py",
    CURRENT_DIR / "10_run_p1_checks.py",
    CURRENT_DIR / "test_dedup_check.py",
]

REQUIRED_FILES = [
    PROCESSED_DIR / "extracted_data.jsonl",
    PROCESSED_DIR / "clean_data.jsonl",
    PROCESSED_DIR / "deduplicated_data.jsonl",
    PROCESSED_DIR / "data_en.jsonl",
    PROCESSED_DIR / "data_zh.jsonl",
    PROCESSED_DIR / "final_data_en.jsonl",
    PROCESSED_DIR / "final_data_zh.jsonl",
    PROCESSED_DIR / "final_data.jsonl",
    PROCESSED_DIR / "quality_filter_stats.json",
    TRAINING_DIR / "serialized_dataset.jsonl",
    TRAINING_DIR / "train.jsonl",
    TRAINING_DIR / "val.jsonl",
    TRAINING_DIR / "smoke_test.jsonl",
    TRAINING_DIR / "training_manifest.json",
    REPORT_DIR / "p1_metrics.json",
    REPORT_DIR / "p1_report.md",
]


def count_lines(path: Path) -> int:
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for _ in f)


def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))
    return records


def run_command(name: str, cmd: list[str], cwd: Path) -> dict:
    completed = subprocess.run(
        cmd,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        check=False,
    )
    return {
        "name": name,
        "command": cmd,
        "returncode": completed.returncode,
        "passed": completed.returncode == 0,
        "stdout": completed.stdout.strip(),
        "stderr": completed.stderr.strip(),
    }


def build_command_checks() -> list[dict]:
    compile_cmd = [sys.executable, "-m", "py_compile", *[str(path) for path in SCRIPT_FILES]]
    checks = [
        run_command("py_compile", compile_cmd, PROJECT_ROOT),
        run_command("dedup_unit_check", [sys.executable, str(CURRENT_DIR / "test_dedup_check.py")], PROJECT_ROOT),
        run_command("training_smoke_test", [sys.executable, str(CURRENT_DIR / "9_training_smoke_test.py")], PROJECT_ROOT),
        run_command("dataset_evaluation", [sys.executable, str(CURRENT_DIR / "8_evaluate_dataset.py")], PROJECT_ROOT),
    ]
    return checks


def build_dataset_checks() -> list[dict]:
    checks = []

    missing_files = [str(path.relative_to(PROJECT_ROOT)) for path in REQUIRED_FILES if not path.exists()]
    checks.append(
        {
            "name": "required_files_exist",
            "passed": not missing_files,
            "details": {"missing_files": missing_files},
        }
    )

    if missing_files:
        return checks

    manifest = load_json(TRAINING_DIR / "training_manifest.json")
    metrics = load_json(REPORT_DIR / "p1_metrics.json")
    quality_stats = load_json(PROCESSED_DIR / "quality_filter_stats.json")

    final_records = load_jsonl(PROCESSED_DIR / "final_data.jsonl")
    final_en_records = load_jsonl(PROCESSED_DIR / "final_data_en.jsonl")
    final_zh_records = load_jsonl(PROCESSED_DIR / "final_data_zh.jsonl")
    train_records = load_jsonl(TRAINING_DIR / "train.jsonl")
    val_records = load_jsonl(TRAINING_DIR / "val.jsonl")
    smoke_records = load_jsonl(TRAINING_DIR / "smoke_test.jsonl")

    final_langs = Counter(record.get("lang", "unknown") for record in final_records)
    train_hashes = {record["text_sha1"] for record in train_records}
    val_hashes = {record["text_sha1"] for record in val_records}
    smoke_ids = {record["id"] for record in smoke_records}
    train_ids = {record["id"] for record in train_records}
    all_final_hashes = [record["text_sha1"] for record in final_records]

    checks.append(
        {
            "name": "final_file_count_matches_language_splits",
            "passed": len(final_records) == len(final_en_records) + len(final_zh_records),
            "details": {
                "final": len(final_records),
                "final_en": len(final_en_records),
                "final_zh": len(final_zh_records),
            },
        }
    )
    checks.append(
        {
            "name": "training_manifest_counts_match_files",
            "passed": (
                manifest.get("num_records") == len(final_records)
                and manifest.get("num_train_records") == len(train_records)
                and manifest.get("num_val_records") == len(val_records)
                and manifest.get("num_smoke_test_records") == len(smoke_records)
            ),
            "details": {
                "manifest_num_records": manifest.get("num_records"),
                "actual_num_records": len(final_records),
                "manifest_num_train_records": manifest.get("num_train_records"),
                "actual_num_train_records": len(train_records),
                "manifest_num_val_records": manifest.get("num_val_records"),
                "actual_num_val_records": len(val_records),
                "manifest_num_smoke_test_records": manifest.get("num_smoke_test_records"),
                "actual_num_smoke_test_records": len(smoke_records),
            },
        }
    )
    checks.append(
        {
            "name": "train_val_no_overlap",
            "passed": not (train_hashes & val_hashes),
            "details": {"overlap_count": len(train_hashes & val_hashes)},
        }
    )
    checks.append(
        {
            "name": "smoke_test_is_subset_of_train",
            "passed": smoke_ids <= train_ids,
            "details": {
                "smoke_records": len(smoke_ids),
                "missing_from_train": sorted(smoke_ids - train_ids)[:10],
            },
        }
    )
    checks.append(
        {
            "name": "final_dataset_no_exact_duplicates",
            "passed": len(all_final_hashes) == len(set(all_final_hashes)),
            "details": {
                "final_records": len(all_final_hashes),
                "unique_hashes": len(set(all_final_hashes)),
            },
        }
    )
    checks.append(
        {
            "name": "metrics_match_final_dataset",
            "passed": (
                metrics.get("stage_counts", {}).get("final") == len(final_records)
                and metrics.get("final_summary", {}).get("languages") == dict(final_langs)
            ),
            "details": {
                "metrics_final_count": metrics.get("stage_counts", {}).get("final"),
                "actual_final_count": len(final_records),
                "metrics_languages": metrics.get("final_summary", {}).get("languages"),
                "actual_languages": dict(final_langs),
            },
        }
    )
    checks.append(
        {
            "name": "quality_filter_stats_match_outputs",
            "passed": (
                quality_stats.get("en", {}).get("kept") == len(final_en_records)
                and quality_stats.get("zh", {}).get("kept") == len(final_zh_records)
            ),
            "details": {
                "quality_en_kept": quality_stats.get("en", {}).get("kept"),
                "actual_final_en": len(final_en_records),
                "quality_zh_kept": quality_stats.get("zh", {}).get("kept"),
                "actual_final_zh": len(final_zh_records),
            },
        }
    )
    checks.append(
        {
            "name": "smoke_test_covers_multiple_languages",
            "passed": len({record.get("lang", "unknown") for record in smoke_records}) >= 2,
            "details": {
                "smoke_languages": dict(Counter(record.get("lang", "unknown") for record in smoke_records))
            },
        }
    )
    checks.append(
        {
            "name": "stage_counts_are_monotonic",
            "passed": (
                count_lines(PROCESSED_DIR / "extracted_data.jsonl")
                >= count_lines(PROCESSED_DIR / "clean_data.jsonl")
                >= count_lines(PROCESSED_DIR / "deduplicated_data.jsonl")
                >= count_lines(PROCESSED_DIR / "final_data.jsonl")
            ),
            "details": {
                "extracted": count_lines(PROCESSED_DIR / "extracted_data.jsonl"),
                "clean": count_lines(PROCESSED_DIR / "clean_data.jsonl"),
                "deduplicated": count_lines(PROCESSED_DIR / "deduplicated_data.jsonl"),
                "final": count_lines(PROCESSED_DIR / "final_data.jsonl"),
            },
        }
    )

    return checks


def render_markdown(results: dict) -> str:
    lines = [
        "# P1 Test Report",
        "",
        f"- Timestamp: {results['timestamp_utc']}",
        f"- Overall status: {'PASS' if results['overall_passed'] else 'FAIL'}",
        f"- Passed checks: {results['passed_checks']}/{results['total_checks']}",
        "",
        "## Command Checks",
        "",
    ]

    for check in results["command_checks"]:
        status = "PASS" if check["passed"] else "FAIL"
        lines.append(f"- {check['name']}: {status}")
        lines.append(f"  - Command: `{' '.join(check['command'])}`")
        if check["stdout"]:
            lines.append(f"  - Stdout: `{check['stdout'][:400]}`")
        if check["stderr"]:
            lines.append(f"  - Stderr: `{check['stderr'][:400]}`")

    lines.extend(["", "## Dataset Checks", ""])
    for check in results["dataset_checks"]:
        status = "PASS" if check["passed"] else "FAIL"
        lines.append(f"- {check['name']}: {status}")
        lines.append(f"  - Details: `{json.dumps(check['details'], ensure_ascii=False)}`")

    lines.extend(["", "## Summary", ""])
    lines.append(f"- Final dataset size: {results['summary']['final_records']}")
    lines.append(f"- Train/val size: {results['summary']['train_records']}/{results['summary']['val_records']}")
    lines.append(f"- Smoke test size: {results['summary']['smoke_test_records']}")
    lines.append(f"- Final language distribution: `{json.dumps(results['summary']['final_languages'], ensure_ascii=False)}`")
    return "\n".join(lines) + "\n"


def main() -> None:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    command_checks = build_command_checks()
    dataset_checks = build_dataset_checks()
    all_checks = command_checks + dataset_checks

    final_records = load_jsonl(PROCESSED_DIR / "final_data.jsonl") if (PROCESSED_DIR / "final_data.jsonl").exists() else []
    train_records = load_jsonl(TRAINING_DIR / "train.jsonl") if (TRAINING_DIR / "train.jsonl").exists() else []
    val_records = load_jsonl(TRAINING_DIR / "val.jsonl") if (TRAINING_DIR / "val.jsonl").exists() else []
    smoke_records = load_jsonl(TRAINING_DIR / "smoke_test.jsonl") if (TRAINING_DIR / "smoke_test.jsonl").exists() else []

    results = {
        "timestamp_utc": datetime.now(UTC).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "overall_passed": all(check["passed"] for check in all_checks),
        "total_checks": len(all_checks),
        "passed_checks": sum(1 for check in all_checks if check["passed"]),
        "command_checks": command_checks,
        "dataset_checks": dataset_checks,
        "summary": {
            "final_records": len(final_records),
            "train_records": len(train_records),
            "val_records": len(val_records),
            "smoke_test_records": len(smoke_records),
            "final_languages": dict(Counter(record.get("lang", "unknown") for record in final_records)),
        },
    }

    with RESULTS_JSON.open("w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    with RESULTS_MD.open("w", encoding="utf-8") as f:
        f.write(render_markdown(results))

    print(json.dumps(results, ensure_ascii=False, indent=2))
    if not results["overall_passed"]:
        raise SystemExit(1)


if __name__ == "__main__":
    main()


### `src/pipeline_utils.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
from __future__ import annotations

import hashlib
import re
from statistics import median
from urllib.parse import urlparse

CJK_RE = re.compile(r"[\u3400-\u4dbf\u4e00-\u9fff\uf900-\ufaff]")
LATIN_RE = re.compile(r"[A-Za-z]")
DIGIT_RE = re.compile(r"\d")
WORD_RE = re.compile(r"[A-Za-z0-9']+")
WHITESPACE_RE = re.compile(r"[ \t\r\f\v]+")
MULTI_NEWLINE_RE = re.compile(r"\n{3,}")


def normalize_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n").replace("\x00", " ")

    cleaned_lines = []
    previous_line = None
    for raw_line in text.split("\n"):
        line = WHITESPACE_RE.sub(" ", raw_line).strip()
        if not line:
            if cleaned_lines and cleaned_lines[-1] != "":
                cleaned_lines.append("")
            previous_line = ""
            continue

        if line == previous_line:
            continue

        cleaned_lines.append(line)
        previous_line = line

    normalized = "\n".join(cleaned_lines).strip()
    return MULTI_NEWLINE_RE.sub("\n\n", normalized)


def sha1_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()


def extract_domain(url: str) -> str:
    if not url:
        return "unknown"

    parsed = urlparse(url)
    host = parsed.netloc.lower().strip()
    if host.startswith("www."):
        host = host[4:]
    return host or "unknown"


def estimate_token_count(text: str) -> int:
    cjk_count = len(CJK_RE.findall(text))
    latin_words = len(WORD_RE.findall(text))
    return cjk_count + latin_words


def script_counts(text: str) -> dict[str, int]:
    return {
        "cjk": len(CJK_RE.findall(text)),
        "latin": len(LATIN_RE.findall(text)),
        "digits": len(DIGIT_RE.findall(text)),
    }


def split_nonempty_lines(text: str) -> list[str]:
    return [line.strip() for line in text.splitlines() if line.strip()]


def duplicate_line_ratio(lines: list[str]) -> float:
    if not lines:
        return 0.0
    unique_count = len(set(lines))
    return 1.0 - unique_count / len(lines)


def short_line_ratio(lines: list[str], max_len: int = 32) -> float:
    if not lines:
        return 0.0
    return sum(1 for line in lines if len(line) <= max_len) / len(lines)


def median_line_length(lines: list[str]) -> float:
    if not lines:
        return 0.0
    return float(median(len(line) for line in lines))


def character_ratio(text: str, charset: str) -> float:
    if not text:
        return 0.0
    if charset == "digits":
        count = len(DIGIT_RE.findall(text))
    elif charset == "latin":
        count = len(LATIN_RE.findall(text))
    elif charset == "cjk":
        count = len(CJK_RE.findall(text))
    else:
        raise ValueError(f"Unsupported charset: {charset}")
    return count / len(text)


### `src/test_dedup_check.py` - 辅助脚本

下面是文件的完整源码。这个单元以源码展示为主，默认不建议直接逐格执行。


In [ ]:
import hashlib

try:
    from datasketch import MinHash, MinHashLSH
except ImportError:
    MinHash = None
    MinHashLSH = None


data_a = "Deep learning is a subset of machine learning using neural networks."
data_b = "Deep learning is a subset of machine learning using neural networks."
data_c = "Cooking pasta requires boiling water and adding salt."


def fallback_check():
    sha_a = hashlib.sha1(data_a.encode("utf-8")).hexdigest()
    sha_b = hashlib.sha1(data_b.encode("utf-8")).hexdigest()
    sha_c = hashlib.sha1(data_c.encode("utf-8")).hexdigest()

    assert sha_a == sha_b, "Identical texts should produce the same digest"
    assert sha_a != sha_c, "Different texts should produce different digests"

    print("datasketch unavailable; fallback exact-duplicate sanity check passed.")
    print({"same_text_same_hash": True, "different_text_different_hash": True})


def minhash_check():
    def get_minhash(text):
        m = MinHash(num_perm=128)
        for word in text.split():
            m.update(word.encode("utf8"))
        return m

    mh_a = get_minhash(data_a)
    mh_b = get_minhash(data_b)
    mh_c = get_minhash(data_c)

    lsh = MinHashLSH(threshold=0.8, num_perm=128)
    lsh.insert("doc_a", mh_a)

    result_b = lsh.query(mh_b)
    result_c = lsh.query(mh_c)

    assert result_b == ["doc_a"], f"Expected duplicate match for B, got: {result_b}"
    assert result_c == [], f"Expected no duplicate match for C, got: {result_c}"

    print(f"查询 B (应该重复): {result_b}")
    print(f"查询 C (不该重复): {result_c}")


if __name__ == "__main__":
    if MinHash is None or MinHashLSH is None:
        fallback_check()
    else:
        minhash_check()


## 三轮实验复现

下面补充的是可直接运行的三轮实验单元，不再只是源码展示。

运行前提：

- 建议在 `p1-mini-c4` 环境中打开这个 notebook
- 默认项目根目录为 `../project_1_mini_c4`
- 三轮实验的归档结果会额外写入 `data/experiments/`，避免和讲义阅读时的历史产物混淆


In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path('/home/xuxin/book/project_1_mini_c4')
EXPERIMENT_ROOT = PROJECT_ROOT / 'data' / 'experiments'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
TRAINING_DIR = PROJECT_ROOT / 'data' / 'training'


def run_step(args):
    print(f"$ {' '.join(args)}")
    completed = subprocess.run(
        args,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)
    return completed


def count_jsonl(path: Path) -> int:
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for _ in f)


def snapshot(paths, target_dir: Path):
    target_dir.mkdir(parents=True, exist_ok=True)
    for path in paths:
        shutil.copy2(path, target_dir / path.name)


EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
print({'project_root': str(PROJECT_ROOT), 'python': sys.executable})


### 实验一：仅做正文抽取，不加严格清洗

这一轮只运行 `src/2_process_warc.py`，目标是得到最初的正文候选集。运行完成后会把 `extracted_data.jsonl` 归档到 `data/experiments/exp1_extraction/`。


In [ ]:
run_step([sys.executable, 'src/2_process_warc.py'])

exp1_dir = EXPERIMENT_ROOT / 'exp1_extraction'
snapshot([PROCESSED_DIR / 'extracted_data.jsonl'], exp1_dir)

exp1_count = count_jsonl(PROCESSED_DIR / 'extracted_data.jsonl')
print({'experiment': 'exp1_extraction', 'extracted_records': exp1_count, 'archive_dir': str(exp1_dir)})


### 实验二：加入启发式清洗与去重

这一轮继续运行 `src/3_clean_data.py` 和 `src/4_deduplicate.py`。实验完成后会归档 `clean_data.jsonl` 与 `deduplicated_data.jsonl`，方便和实验一做数量对比。


In [ ]:
run_step([sys.executable, 'src/3_clean_data.py'])
run_step([sys.executable, 'src/4_deduplicate.py'])

exp2_dir = EXPERIMENT_ROOT / 'exp2_clean_dedup'
snapshot(
    [
        PROCESSED_DIR / 'clean_data.jsonl',
        PROCESSED_DIR / 'deduplicated_data.jsonl',
    ],
    exp2_dir,
)

exp2_summary = {
    'experiment': 'exp2_clean_dedup',
    'clean_records': count_jsonl(PROCESSED_DIR / 'clean_data.jsonl'),
    'deduplicated_records': count_jsonl(PROCESSED_DIR / 'deduplicated_data.jsonl'),
    'archive_dir': str(exp2_dir),
}
print(exp2_summary)


### 实验三：加入语言拆分与 KenLM 过滤

这一轮运行 `src/5_split_lang.py` 和 `src/6_quality_filter.py`，并把语种切分结果、最终过滤结果和 `quality_filter_stats.json` 一起归档到 `data/experiments/exp3_lang_quality/`。


In [ ]:
run_step([sys.executable, 'src/5_split_lang.py'])
run_step([sys.executable, 'src/6_quality_filter.py'])

exp3_dir = EXPERIMENT_ROOT / 'exp3_lang_quality'
snapshot(
    [
        PROCESSED_DIR / 'data_en.jsonl',
        PROCESSED_DIR / 'data_zh.jsonl',
        PROCESSED_DIR / 'final_data_en.jsonl',
        PROCESSED_DIR / 'final_data_zh.jsonl',
        PROCESSED_DIR / 'final_data.jsonl',
        PROCESSED_DIR / 'quality_filter_stats.json',
    ],
    exp3_dir,
)

quality_stats = json.loads((PROCESSED_DIR / 'quality_filter_stats.json').read_text(encoding='utf-8'))
exp3_summary = {
    'experiment': 'exp3_lang_quality',
    'en_candidates': count_jsonl(PROCESSED_DIR / 'data_en.jsonl'),
    'zh_candidates': count_jsonl(PROCESSED_DIR / 'data_zh.jsonl'),
    'final_en_records': count_jsonl(PROCESSED_DIR / 'final_data_en.jsonl'),
    'final_zh_records': count_jsonl(PROCESSED_DIR / 'final_data_zh.jsonl'),
    'final_total_records': count_jsonl(PROCESSED_DIR / 'final_data.jsonl'),
    'quality_stats_file': str(exp3_dir / 'quality_filter_stats.json'),
}
print(exp3_summary)
print(json.dumps(quality_stats, ensure_ascii=False, indent=2))


### 实验结果汇总

这个单元把三轮实验的关键数量汇总成一张表，便于直接截图或写入报告。


In [ ]:
summary = [
    {
        'stage': 'exp1_extraction',
        'count': count_jsonl(EXPERIMENT_ROOT / 'exp1_extraction' / 'extracted_data.jsonl'),
    },
    {
        'stage': 'exp2_clean',
        'count': count_jsonl(EXPERIMENT_ROOT / 'exp2_clean_dedup' / 'clean_data.jsonl'),
    },
    {
        'stage': 'exp2_deduplicated',
        'count': count_jsonl(EXPERIMENT_ROOT / 'exp2_clean_dedup' / 'deduplicated_data.jsonl'),
    },
    {
        'stage': 'exp3_final',
        'count': count_jsonl(EXPERIMENT_ROOT / 'exp3_lang_quality' / 'final_data.jsonl'),
    },
]

for row in summary:
    print(row)
